In [1]:
import zipfile

with zipfile.ZipFile("/content/idflakies-output.zip", "r") as z:
    z.extractall("/content/idflakies-output")

In [3]:
import glob

json_files = glob.glob(
    "/content/idflakies-output/**/detection-results/**/*.json",
    recursive=True
)

print("JSON files length:", len(json_files))

for f in json_files[:15]:
    print(f)

JSON files length: 257
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/flaky-lists.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/reverse-class/round0.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/original/round2.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/original/round9.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/original/round1.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/original/round3.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/original/round5.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/original/round0.json
/content/idflakies-output/all-output/kevinsawicki.http-request_output/li

In [4]:
import json

round_files = glob.glob(
    "/content/idflakies-output/**/detection-results/**/round*.json",
    recursive=True
)

print("Round files length:", len(round_files))

first_path = round_files[0]

print("Looking at:", first_path)

with open(first_path, "r") as f:
    data = json.load(f)
print("\nType:", type(data))

Round files length: 48
Looking at: /content/idflakies-output/all-output/kevinsawicki.http-request_output/lib/detection-results/reverse-class/round0.json

Type: <class 'dict'>


In [5]:
print("Fields for first round:")

if isinstance(data, dict):
    for key in data.keys():
        print("-", key)

Fields for first round:
- testRunIds
- unfilteredTests
- filteredTests
- roundTime


## See what type each field is

In [7]:
for key, value in data.items():
    print(key, ":", type(value).__name__)

testRunIds : list
unfilteredTests : dict
filteredTests : dict
roundTime : float


## Inspect flaky-lists.json

In [9]:
flaky_lists_path = glob.glob(
    "/content/idflakies-output/**/detection-results/flaky-lists.json",
    recursive=True)[0]

with open(flaky_lists_path, "r") as f:
    flaky_lists_data = json.load(f)

print(type(flaky_lists_data))

if isinstance(flaky_lists_data, dict):
    for key, value in flaky_lists_data.items():
        print(key, ":", type(value).__name__)

<class 'dict'>
dts : list


In [11]:
import os

# Prevent duplicates
detectors = set()

for path in round_files:
    detector = os.path.basename(os.path.dirname(path))
    detectors.add(detector)

print("Type of flaky detectors:")

for detector in sorted(detectors):
    print(detector)

Type of flaky detectors:
original
random
random-class
random-class-confirmation-sampling
random-class-verify
random-confirmation-sampling
random-verify
reverse
reverse-class
reverse-verify


### Detector Configurations Generated

The execution gave us 10 detector/configuration types used to determine/check flaky behavior --

- `original`
- `random`
- `random-class`
- `random-class-confirmation-sampling`
- `random-class-verify`
- `random-confirmation-sampling`
- `random-verify`
- `reverse`
- `reverse-class`
- `reverse-verify`

These detector types show up in the  `round_.json` files.

## Look into one specific json file (round0.json)

In [14]:
r0 = glob.glob(
    "/content/idflakies-output/**/detection-results/original/round0.json",
    recursive=True)[0]

with open(r0, "r") as f:
    round_data = json.load(f)

print("Attributes for round0.json:")
for key, value in round_data.items():
    print(f"{key}: {type(value).__name__}")

Attributes for round0.json:
testRunIds: list
unfilteredTests: dict
filteredTests: dict
roundTime: float


In [17]:
print("First unfiltered test:")
print(next(iter(round_data["unfilteredTests"].items()), None))

print("\nFirst filtered test:")
print(next(iter(round_data["filteredTests"].items()), None))

First unfiltered test:
('dts', [])

First filtered test:
('dts', [])


## Open all_flaky_tests_list.csv

In [15]:
import pandas as pd

flaky_csv = glob.glob(
    "/content/idflakies-output/**/all_flaky_tests_list.csv",
    recursive=True
)[0]

print("Flaky File:", flaky_csv)

df_flaky = pd.read_csv(flaky_csv, header=None)
print("Shape:", df_flaky.shape)
df_flaky.head(20)

Flaky File: /content/idflakies-output/all-output/kevinsawicki.http-request_output/all_flaky_tests_list.csv
Shape: (80, 2)


,0,1
0,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random-class/round1.json
1,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random-class/round3.json
2,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random/round1.json
3,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random/round3.json
4,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/reverse/round0.json
5,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random-class/round1.json
6,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random-class/round3.json
7,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random/round1.json
8,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random/round3.json
9,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/reverse/round0.json


In [16]:
df_flaky.columns = ["test_name", "detection_result_path"]
df_flaky.head(20)

,test_name,detection_result_path
0,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random-class/round1.json
1,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random-class/round3.json
2,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random/round1.json
3,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random/round3.json
4,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/reverse/round0.json
5,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random-class/round1.json
6,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random-class/round3.json
7,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random/round1.json
8,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/random/round3.json
9,com.github.kevinsawicki.http.HttpRequestTest.d...,./lib/detection-results/reverse/round0.json


## Follow 1 row from all_flaky_tests_list.csv

In [19]:
df_flaky.columns = ["test_name", "detection_result_path"]

first_row = df_flaky.iloc[0]

print("Test name:")
print(first_row["test_name"])

print("\nDetection JSON path:")
print(first_row["detection_result_path"])

Test name:
com.github.kevinsawicki.http.HttpRequestTest.basicProxyAuthentication

Detection JSON path:
./lib/detection-results/random-class/round1.json


## Open that path

In [21]:
base_path = "/content/idflakies-output/all-output/kevinsawicki.http-request_output"

json_path = base_path + first_row["detection_result_path"][1:]

with open(json_path, "r") as f:
    detected_round = json.load(f)

print("\nUnfiltered tests:")
print(detected_round["unfilteredTests"])

print("\nFiltered tests:")
print(detected_round["filteredTests"])


Unfiltered tests:
{'dts': [{'name': 'com.github.kevinsawicki.http.HttpRequestTest.deleteWithMappedQueryParams', 'intended': {'order': ['com.github.kevinsawicki.http.EncodeTest.encode', 'com.github.kevinsawicki.http.EncodeTest.encodeMalformedUri', 'com.github.kevinsawicki.http.HttpRequestTest.acceptCharsetHeader', 'com.github.kevinsawicki.http.HttpRequestTest.acceptEncodingHeader', 'com.github.kevinsawicki.http.HttpRequestTest.acceptHeader', 'com.github.kevinsawicki.http.HttpRequestTest.acceptJson', 'com.github.kevinsawicki.http.HttpRequestTest.appendArrayQueryParams', 'com.github.kevinsawicki.http.HttpRequestTest.appendEmptyMappedQueryParams', 'com.github.kevinsawicki.http.HttpRequestTest.appendEmptyVarargsQueryParams', 'com.github.kevinsawicki.http.HttpRequestTest.appendListQueryParams', 'com.github.kevinsawicki.http.HttpRequestTest.appendMappedQueryParamsWithExistingParams', 'com.github.kevinsawicki.http.HttpRequestTest.appendMappedQueryParamsWithExistingQueryStart', 'com.github.kev

In [22]:
print("Unfiltered DTs:",
      len(detected_round["unfilteredTests"]["dts"]))

print("Filtered DTs:",
      len(detected_round["filteredTests"]["dts"]))

Unfiltered DTs: 16
Filtered DTs: 16


In [23]:
first_test = detected_round["unfilteredTests"]["dts"][0]
print(first_test.keys())

dict_keys(['name', 'intended', 'revealed', 'type'])


In [24]:
# Investigate the nested headers for detected_round
print("intended:", first_test["intended"].keys())

intended: dict_keys(['order', 'result', 'testRunId'])


In [25]:
print("revealed:", first_test["revealed"].keys())

revealed: dict_keys(['order', 'result', 'testRunId'])


## Print first test out for easier analysis of what information we are provided with

In [28]:
print("Test:", first_test["name"])
print("Type:", first_test["type"])

print("\nIntended result:", first_test["intended"]["result"])
print("Intended test run ID:", first_test["intended"]["testRunId"])

print("\nRevealed result:", first_test["revealed"]["result"])
print("Revealed test run ID:", first_test["revealed"]["testRunId"])

print("\nIntended order length:",
      len(first_test["intended"]["order"]))

print("Revealed order length:",
      len(first_test["revealed"]["order"]))

Test: com.github.kevinsawicki.http.HttpRequestTest.deleteWithMappedQueryParams
Type: OD

Intended result: ERROR
Intended test run ID: 1791067828796-9cabacfa-64b2-4271-bb5e-1fe16ab8e639

Revealed result: PASS
Revealed test run ID: 1791067830728-cac5aa53-0104-4c9a-bd6b-3c30c2aa5f2e

Intended order length: 40
Revealed order length: 122


In [29]:
rows = []

for test in detected_round["unfilteredTests"]["dts"]:
    rows.append({
        "test_name": test["name"],
        "type": test["type"],
        "intended_result": test["intended"]["result"],
        "revealed_result": test["revealed"]["result"],
        "intended_order_size": len(test["intended"]["order"]),
        "revealed_order_size": len(test["revealed"]["order"])
    })

round_df = pd.DataFrame(rows)

round_df

,test_name,type,intended_result,revealed_result,intended_order_size,revealed_order_size
0,com.github.kevinsawicki.http.HttpRequestTest.d...,OD,ERROR,PASS,40,122
1,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,ERROR,PASS,77,85
2,com.github.kevinsawicki.http.HttpRequestTest.h...,OD,ERROR,PASS,85,77
3,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,FAILURE,PASS,75,87
4,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,FAILURE,PASS,73,89
5,com.github.kevinsawicki.http.HttpRequestTest.h...,OD,ERROR,PASS,87,75
6,com.github.kevinsawicki.http.HttpRequestTest.d...,OD,ERROR,PASS,41,121
7,com.github.kevinsawicki.http.HttpRequestTest.h...,OD,ERROR,PASS,84,78
8,com.github.kevinsawicki.http.HttpRequestTest.h...,OD,ERROR,PASS,86,76
9,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,ERROR,PASS,78,84
